# GeoAtt-PointNet++ — Phase 2: Compare & Analyze

Post-hoc analysis dari hasil 4-variant fair ablation. **CPU only** — jalankan di Colab setelah notebook 01 selesai (atau lokal jika `EVAL_DIR` ter-sync dari Drive).

**Tujuan utama:** menguji hipotesis bahwa **GeoAtt memberikan dampak signifikan** terhadap identifikasi telapak tangan, lewat paired t-test `with_geom` vs `no_geom` per seed.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

PROJECT_ROOT = Path("/content/drive/MyDrive/3DCNN")
EVAL_DIR   = PROJECT_ROOT / "eval_results"
OUTPUT_DIR = PROJECT_ROOT / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.compare_utils import (
    load_all_results,
    aggregate_by_variant,
    format_mean_std,
    paired_ttest,
    export_latex_table,
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

VARIANTS = ["no_geom", "gam_only", "fuse_only", "with_geom"]
SEEDS    = [7, 42, 123, 2026, 31337]

print(f"EVAL_DIR:   {EVAL_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


## 2. Load Results

In [ ]:
df_test    = load_all_results(EVAL_DIR, holdout=False)
df_holdout = load_all_results(EVAL_DIR, holdout=True)

print("Test results per variant:")
print(df_test.groupby("variant").size())
print()
print("Holdout results per variant:")
print(df_holdout.groupby("variant").size())


## 3. Aggregation Table (Mean ± Std)

In [ ]:
metrics = ["eer", "auc", "tar_at_far1", "tar_at_far01", "dprime", "rank1", "rank5", "rank10"]

print("=== TEST SET ===")
for m in metrics:
    print(f"\n{m.upper()}:")
    print(format_mean_std(df_test, m))

print("\n=== HOLDOUT SET ===")
for m in metrics:
    print(f"\n{m.upper()}:")
    print(format_mean_std(df_holdout, m))


## 4. Boxplots — Test Set

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
plot_metrics = ["eer", "auc", "tar_at_far1", "tar_at_far01", "dprime", "rank1"]
for ax, metric in zip(axes.flat, plot_metrics):
    sns.boxplot(data=df_test, x="variant", y=metric, ax=ax, order=VARIANTS)
    ax.set_title(metric.upper())
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("Test Set — 4 Variant Comparison (5 seeds)", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "boxplots_test.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Boxplots — Holdout Set

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, metric in zip(axes.flat, plot_metrics):
    sns.boxplot(data=df_holdout, x="variant", y=metric, ax=ax, order=VARIANTS)
    ax.set_title(metric.upper())
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("Holdout Set — 4 Variant Comparison (5 seeds)", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "boxplots_holdout.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Statistical Tests — Uji Hipotesis Utama

Paired t-test antar variant per seed. Hipotesis utama: `with_geom` vs `no_geom`.

In [ ]:
test_pairs = [
    ("with_geom", "no_geom"),    # Hipotesis utama: full GeoAtt vs PointNet++ murni
    ("with_geom", "gam_only"),   # Kontribusi fusion di atas GAM
    ("with_geom", "fuse_only"),  # Kontribusi GAM di atas fusion
    ("gam_only",  "no_geom"),    # Kontribusi GAM saja
    ("fuse_only", "no_geom"),    # Kontribusi fusion saja
]

def run_ttests(df, label):
    print(f"=== {label} — Paired T-Tests ===")
    for va, vb in test_pairs:
        print(f"\n[{va} vs {vb}]")
        for metric in ["eer", "auc", "tar_at_far1", "rank1"]:
            t, p, d = paired_ttest(df, va, vb, metric)
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            print(f"  {metric:12s}: t={t:7.3f}, p={p:.4f} [{sig}], Cohen's d={d:.3f}")

run_ttests(df_test,    "TEST SET")
print()
run_ttests(df_holdout, "HOLDOUT SET")


## 7. CMC Curve — Identification 1:N

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ranks = [1, 5, 10]
rank_cols = ["rank1", "rank5", "rank10"]

for variant in VARIANTS:
    sub = df_test[df_test["variant"] == variant]
    if len(sub) == 0:
        continue
    means = [sub[c].mean() for c in rank_cols]
    stds  = [sub[c].std()  for c in rank_cols]
    ax.errorbar(ranks, means, yerr=stds, marker="o", capsize=4, linewidth=2, label=variant)

ax.set_xlabel("Rank")
ax.set_ylabel("Recognition Rate")
ax.set_title("CMC Curve — Test Set (mean ± std across seeds)")
ax.set_xticks(ranks)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cmc_curve.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Pairwise Variant Metric Heatmap

In [ ]:
# Heatmap rangkuman: baris = variant, kolom = metric. Mean across seeds.
heatmap_metrics = ["eer", "auc", "tar_at_far1", "tar_at_far01", "rank1", "rank5"]
mat = np.array([
    [df_test[df_test["variant"] == v][m].mean() for m in heatmap_metrics]
    for v in VARIANTS
])
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(
    mat, annot=True, fmt=".4f", cmap="RdYlGn",
    xticklabels=[m.upper() for m in heatmap_metrics],
    yticklabels=VARIANTS, ax=ax, cbar_kws={"label": "Mean across seeds"},
)
ax.set_title("Test Set — Variant × Metric (mean across seeds)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "variant_metric_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Similarity Distribution — Genuine vs Impostor

Load file `*_scores.npz` yang ditulis oleh `evaluate.py --save_scores`. Cell ini di-skip otomatis kalau `.npz` belum ada.

Visual ini penting untuk defense: kalau GeoAtt memang signifikan, distribusi genuine vs impostor seharusnya lebih ter-separasi di `with_geom` daripada di `no_geom`.

In [ ]:
from sklearn.metrics import roc_curve

def compute_eer_threshold(labels, scores):
    fpr, tpr, thr = roc_curve(labels, scores)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return thr[idx], (fpr[idx] + fnr[idx]) / 2.0

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flat
rep_seed = SEEDS[0] if SEEDS else 42

for ax, variant in zip(axes, VARIANTS):
    npz_path = EVAL_DIR / variant / f"seed_{rep_seed}" / f"{variant}_scores.npz"
    if not npz_path.exists():
        ax.set_title(f"{variant} — .npz missing")
        ax.set_axis_off()
        continue
    data    = np.load(npz_path)
    labels  = data["labels"]
    scores  = data["scores"]
    genuine = scores[labels == 1]
    impost  = scores[labels == 0]
    eer_thr, eer = compute_eer_threshold(labels, scores)

    ax.hist(genuine, bins=50, alpha=0.6, label=f"Genuine (n={len(genuine)})", color="steelblue", density=True)
    ax.hist(impost,  bins=50, alpha=0.6, label=f"Impostor (n={len(impost)})", color="salmon",   density=True)
    ax.axvline(eer_thr, color="black", linestyle="--", label=f"EER={eer:.3f} @ thr={eer_thr:.3f}")
    ax.set_title(f"{variant} (seed {rep_seed})")
    ax.set_xlabel("Cosine Similarity")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Similarity Distribution — Genuine vs Impostor", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "similarity_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Export Tables

In [ ]:
agg_test = aggregate_by_variant(df_test)
agg_test.to_csv(OUTPUT_DIR / "aggregate_test.csv")
print("Saved:", OUTPUT_DIR / "aggregate_test.csv")

agg_holdout = aggregate_by_variant(df_holdout)
agg_holdout.to_csv(OUTPUT_DIR / "aggregate_holdout.csv")
print("Saved:", OUTPUT_DIR / "aggregate_holdout.csv")

latex_test = export_latex_table(
    df_test,
    metrics=["eer", "auc", "tar_at_far1", "rank1"],
    save_path=OUTPUT_DIR / "table_test.tex",
    caption="Fair Ablation Results on Test Set (mean $\\pm$ std, 5 seeds)",
)
print("Saved:", OUTPUT_DIR / "table_test.tex")

latex_holdout = export_latex_table(
    df_holdout,
    metrics=["eer", "auc", "tar_at_far1", "rank1"],
    save_path=OUTPUT_DIR / "table_holdout.tex",
    caption="Fair Ablation Results on Holdout Set (mean $\\pm$ std, 5 seeds)",
)
print("Saved:", OUTPUT_DIR / "table_holdout.tex")


## 11. Completion

In [ ]:
print("All analysis outputs saved to:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")
